# Étude de rémunération IT — parcours reproductible

Le CSV a une provenance non vérifiée. Les chiffres décrivent le fichier et ne prouvent pas sa représentativité. Ce notebook appelle les mêmes modules que la CLI et Streamlit ; les anciennes sorties ont été retirées. Voir `docs/EXPLICATION_CODE.md` et `docs/DATA_CARD.md`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from salary_study.train import run, ROOT
from salary_study.data import load_data, split_data, FEATURES
from salary_study.inference import load_bundle, predict
from salary_study.explain import explain_predictions


## Audit du fichier
Les doublons exacts sont exclus du calcul, sans modifier le CSV. Les groupes d’entrées identiques restent ensemble.


In [ ]:
data, audit = load_data(ROOT / "DATASET_IT_MAROC_2025.csv")
audit


## Entraînement et sélection
Cinq plis sur train, calibration distincte puis test final. Cette cellule régénère les artefacts et le rapport. La régression from scratch est dans `salary_study/models.py`.


In [ ]:
metadata = run()
pd.read_csv(ROOT / "artifacts/model_comparison.csv")


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((ROOT / "artifacts/rapport_client.md").read_text(encoding="utf-8")))
display(Image(filename=str(ROOT / "artifacts/residual_diagnostics.png")))


## Une simulation expliquée
Exemple tiré du train pour illustrer l’inférence, sans intervenir dans l’évaluation test. Intervalle individuel distinct des IC des métriques.


In [ ]:
bundle, metadata = load_bundle(ROOT / "artifacts")
row = split_data(data)["train"].head(1)[FEATURES]
point, lower, upper = predict(bundle, row)
contributions, base = explain_predictions(bundle["pipeline"], bundle["background"], row)
display(row)
print({"prediction_mad": point[0], "lower_mad": lower[0], "upper_mad": upper[0], "reference_mad": base[0]})
display(contributions)
